# YARP-again demo -- June, 2025

Code version that this demo was written for:
- **branch:** main
- **commit:** d0dc1ff


In order to get the code up and running, install a conda environment for YARP-again dependencies
```
cd yarp-again
conda env create -f environment.yml
conda activate yarp-again
```

Then do a local pip install of YARP-again: `pip install -e .`

If all goes well, you should be able to run the test suite like so: `pytest -v test/`

And you should see 2 out of 14 failing test cases:

```
test/yarpecule/graph/test_smiles.py::TestSmi2Adj::test_rad_explicitH FAILED
test/yarpecule/graph/test_smiles.py::TestSmi2Adj::test_rad_full_map FAILED
```

In [1]:
import yarp as yp

## The basics: SMILES to yarpecule

There are two **modes** to go from SMILES to a yarpecule (adjacency matrix)
- `rdkit`: uses RDKit to generate the adjacency matrix and XYZ coordinates
- `yarp`: uses in-house code to parse SMILES string and generate "adjacency matrix" and XYZ coordinates

Both of these modes return the same stuff, except the "adjacency matrix" generated in `yarp` mode is equivalent to the bond electron matrix generated by `find_lewis` routines.

(Fun fact, the old `find_lewis` code has been ported over to a `lewis_struct` class. I haven't rigorously tested it yet though, so beware!)

In [2]:
ethene_smi = "C=C"

In [3]:
# Using the default RDKit mode to create a yarpecule object
ethene = yp.yarpecule(ethene_smi, canon=False, mode='rdkit')

print(ethene.elements)
print(ethene.adj_mat)
print(ethene.lewis.bond_mats)


['c', 'c', 'h', 'h', 'h', 'h']
[[0. 1. 1. 1. 0. 0.]
 [1. 0. 0. 0. 1. 1.]
 [1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]]
[array([[0., 2., 1., 1., 0., 0.],
       [2., 0., 0., 0., 1., 1.],
       [1., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.]])]


In [4]:
# Using the in-house YARP mode to create a yarpecule object
ethene = yp.yarpecule(ethene_smi, canon=False, mode='yarp')

print(ethene.elements)
print(ethene.adj_mat) # this is basically the same as the bond electron matrix
print(ethene.lewis.bond_mats)

['c', 'c', 'h', 'h', 'h', 'h']
[[0. 2. 1. 1. 0. 0.]
 [2. 0. 0. 0. 1. 1.]
 [1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]]
[array([[0., 2., 1., 1., 0., 0.],
       [2., 0., 0., 0., 1., 1.],
       [1., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.]])]


## Atom mapped SMILES to yarpecule

If you want to get adjacency matrices that preserve the atomic mapping of SMILES strings, then you'll need to use the `yarp` mode to generate the yarpecule object.

**Make sure you turn off the canonicalization routines!!**

In [5]:
haa_canon_smi = 'O=CCO'
haa = yp.yarpecule(haa_canon_smi, canon=False, mode='yarp')

print(haa.elements)
print(haa.adj_mat)
print(haa.lewis.bond_mats)

['o', 'c', 'c', 'o', 'h', 'h', 'h', 'h']
[[0. 2. 0. 0. 0. 0. 0. 0.]
 [2. 0. 1. 0. 1. 0. 0. 0.]
 [0. 1. 0. 1. 0. 1. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]]
[array([[4., 2., 0., 0., 0., 0., 0., 0.],
       [2., 0., 1., 0., 1., 0., 0., 0.],
       [0., 1., 0., 1., 0., 1., 1., 0.],
       [0., 0., 1., 4., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0.]])]


In [6]:
haa_map_smi = '[C:0]([C:1]([O:7][H:11])([H:13])[H:14])(=[O:6])[H:12]'
haa = yp.yarpecule(haa_map_smi, canon=False, mode='yarp')

print(haa.elements)
print(haa.adj_mat)
print(haa.lewis.bond_mats)

['c', 'c', 'o', 'o', 'h', 'h', 'h', 'h']
[[0. 1. 2. 0. 0. 1. 0. 0.]
 [1. 0. 0. 1. 0. 0. 1. 1.]
 [2. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]]
[array([[0., 1., 2., 0., 0., 1., 0., 0.],
       [1., 0., 0., 1., 0., 0., 1., 1.],
       [2., 0., 4., 0., 0., 0., 0., 0.],
       [0., 1., 0., 4., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.]])]
